# SynBioCrow 2.2.0 RC1 — published-release acceptance
This notebook validates the **published GitHub RC1 release**, not the development checkout. It downloads the three attached M9 assets, verifies GitHub-published SHA-256 digests and manifest inventory, installs the source reproducibility ZIP into an isolated virtual environment, runs tests and M7/M9 gates, audits public README/install language, and emits a final acceptance bundle to Google Drive.

It does **not** create or modify the final `v2.2.0` tag/release. Promotion remains a separate explicit release action after this acceptance report is green.


In [ ]:
import os,sys,json,hashlib,subprocess,shutil,zipfile,urllib.request,time,re
from pathlib import Path
from google.colab import drive

OWNER="theiman112860"; REPO="SynBioCrow"; TAG="v2.2.0-rc1"
EXPECTED_COMMIT="8839e83cf299a13513a91adfff6225bc7b5719c6"
ASSETS={
 "SynBioCrow_2.2.0-rc1_PACKAGE_SUMMARY.json":"460e2d8a80c36c10190f91dc611a2c320a65bb6eb7e43b7cb0f17719907f475c",
 "SynBioCrow_2.2.0-rc1_release_manifest.json":"a04ae73f236bb9f6f5608ad9f30fd8719a1a6ce5c56d3a619f6f2e07d8031a05",
 "SynBioCrow_2.2.0-rc1_SOURCE_REPRODUCIBILITY.zip":"ae2e1abc9f469033cf524897abf74705d47e6012cc014f54fcb5ed823a037d14",
}
ROOT=Path("/content/synbiocrow_rc1_acceptance")
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir()
def sha256(p):
 h=hashlib.sha256()
 with open(p,"rb") as f:
  for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
 return h.hexdigest()
def run(cmd,cwd=None,timeout=1800):
 print("$"," ".join(map(str,cmd)),flush=True)
 p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
 print(p.stdout,flush=True)
 if p.returncode: raise RuntimeError(f"command failed rc={p.returncode}: {cmd}")
 return p.stdout
print("Acceptance target",TAG,EXPECTED_COMMIT)


In [ ]:
# 1. Download exactly the published release assets and verify their GitHub-recorded digests.
for name,expected in ASSETS.items():
 url=f"https://github.com/{OWNER}/{REPO}/releases/download/{TAG}/{name}"
 dst=ROOT/name
 print("download",url)
 urllib.request.urlretrieve(url,dst)
 got=sha256(dst)
 print(name,dst.stat().st_size,got)
 assert got==expected,(name,got,expected)
print("PUBLISHED ASSET DIGESTS: PASS")


In [ ]:
# 2. Cross-check package summary, release manifest, commit, ZIP hash, and every inventory entry.
summary=json.loads((ROOT/"SynBioCrow_2.2.0-rc1_PACKAGE_SUMMARY.json").read_text())
manifest=json.loads((ROOT/"SynBioCrow_2.2.0-rc1_release_manifest.json").read_text())
PACKAGE_COMMIT=summary["git_commit"]
assert manifest["git_commit"]==PACKAGE_COMMIT, (manifest["git_commit"],PACKAGE_COMMIT)
print("published package commit",PACKAGE_COMMIT)
print("tagged RC commit",EXPECTED_COMMIT)
# The published M9 package was generated from the content commit immediately
# before the finalization-notebook commit. Validate this explicitly: the tagged
# RC may be ahead only by release-orchestration/notebook files, but must not
# change packaged runtime/source content.
compare_url=f"https://api.github.com/repos/{OWNER}/{REPO}/compare/{PACKAGE_COMMIT}...{EXPECTED_COMMIT}"
with urllib.request.urlopen(compare_url) as resp:
    cmp=json.loads(resp.read().decode("utf-8"))
assert cmp.get("status") in ("ahead","identical"),cmp.get("status")
assert cmp.get("behind_by",0)==0,cmp
allowed_tag_only={"notebooks/SynBioCrow_2_2_RC1_FINALIZE_KNIME_FREE_PINNED_b1eaf11.ipynb"}
changed={x["filename"] for x in cmp.get("files",[])}
unexpected=sorted(changed-allowed_tag_only)
assert not unexpected,f"tag commit changes packaged release content: {unexpected}"
print("PACKAGE COMMIT ANCESTRY / TAG-ONLY DELTA: PASS",sorted(changed))
zip_path=ROOT/summary["source_zip"]
assert sha256(zip_path)==summary["source_zip_sha256"]
src=ROOT/"source"
src.mkdir()
with zipfile.ZipFile(zip_path) as z: z.extractall(src)
bad=[]
for item in manifest["files"]:
 p=src/item["path"]
 if not p.is_file() or p.stat().st_size!=item["bytes"] or sha256(p)!=item["sha256"]:
  bad.append(item["path"])
assert not bad,f"manifest mismatches: {bad[:20]}"
assert manifest["retropath_primary_runtime"]=="retropath_standalone"
assert manifest["retropath_knime_required"] is False
assert manifest["retropath_equivalence_policy"]=="EXACT_COMPOUND_TRANSITION_MULTISET"
print("M9 MANIFEST + INVENTORY: PASS",len(manifest["files"]),"files")


In [ ]:
# 3. Clean-install the published source ZIP in an isolated venv.
venv=ROOT/"venv"
run([sys.executable,"-m","venv",venv])
py=venv/"bin/python"; pip=venv/"bin/pip"
run([py,"-m","pip","install","-U","pip","setuptools","wheel"])
run([pip,"install",str(src)])
run([py,"-c","import synbiocrow; print('synbiocrow import PASS',synbiocrow.__file__)"])
run([venv/"bin/synbiocrow","--help"])
print("CLEAN INSTALL + CLI: PASS")


In [ ]:
# 4. Run repository tests from the immutable release artifact.
# pytest is test tooling, so install it only into the isolated acceptance venv.
run([pip,"install","pytest"])
tests=src/"tests"
if tests.is_dir():
 run([py,"-m","pytest","-q",str(tests)],cwd=src,timeout=1800)
 print("RELEASE TEST SUITE: PASS")
else:
 print("RELEASE TEST SUITE: SKIP (tests directory not present in published artifact)")


In [ ]:
# 5. Run the release's own M7 and M9 gates from the extracted published artifact.
run([py,str(src/"scripts/m7_release_readiness.py")],cwd=src,timeout=1800)
run([py,str(src/"scripts/build_m9_release.py"),"--check"],cwd=src,timeout=1800)
print("M7 + M9 RELEASE GATES: PASS")


In [ ]:
# 6. Audit public README/install contract for the RC.
readme=(src/"README.md").read_text(encoding="utf-8")
checks={
 "version/install package present":("synbiocrow" in readme.lower()),
 "standalone RetroPath documented":("retropath standalone" in readme.lower()),
 "KNIME not primary requirement":(("optional legacy" in readme.lower()) or ("not required" in readme.lower())),
 "BioPKS acknowledgement documented":(("biopks" in readme.lower()) and ("acknowledg" in readme.lower())),
 "scientific lifecycle documented":all(x.lower() in readme.lower() for x in ["Candidate","Mature","Certified"]),
}
print(json.dumps(checks,indent=2))
assert all(checks.values()),checks
print("PUBLIC README CONTRACT: PASS")


In [ ]:
# 7. Verify the committed standalone equivalence certificate carried by the release.
cert=(src/"docs/RETROPATH_STANDALONE_EQUIVALENCE.md").read_text(encoding="utf-8")
required=["EXACT_COMPOUND_TRANSITION_MULTISET","historical rows: 34","standalone rows: 34","precision: 1.0","recall: 1.0","Jaccard: 1.0","KNIME is not required"]
missing=[x for x in required if x.lower() not in cert.lower()]
assert not missing,missing
print("RETROPATH EQUIVALENCE CERTIFICATE: PASS")


In [ ]:
# 8. Emit and persist final RC acceptance evidence.
report={
 "schema":"synbiocrow.rc_acceptance.v1",
 "release":"2.2.0-rc1",
 "tag":TAG,
 "tag_commit":EXPECTED_COMMIT,
 "package_commit":PACKAGE_COMMIT,
 "published_asset_sha256":ASSETS,
 "source_zip_sha256":sha256(zip_path),
 "source_zip_bytes":zip_path.stat().st_size,
 "manifest_file_count":manifest["file_count"],
 "m7_reproducibility_digest":manifest["m7_reproducibility_digest"],
 "retropath_primary_runtime":manifest["retropath_primary_runtime"],
 "retropath_knime_required":manifest["retropath_knime_required"],
 "retropath_equivalence_policy":manifest["retropath_equivalence_policy"],
 "checks":{
   "published_asset_digests":"PASS",
   "manifest_inventory":"PASS",
   "clean_install":"PASS",
   "cli":"PASS",
   "tests":"PASS" if tests.is_dir() else "SKIP_NOT_PACKAGED",
   "m7":"PASS","m9":"PASS","readme_contract":"PASS","retropath_equivalence_certificate":"PASS"
 },
 "acceptance":"PASS"
}
report_path=ROOT/"SynBioCrow_2.2.0-rc1_ACCEPTANCE_REPORT.json"
report_path.write_text(json.dumps(report,indent=2,sort_keys=True)+"\n")
bundle=ROOT/"SynBioCrow_2.2.0-rc1_ACCEPTANCE_BUNDLE.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
 z.write(report_path,report_path.name)
 z.write(ROOT/"SynBioCrow_2.2.0-rc1_PACKAGE_SUMMARY.json")
 z.write(ROOT/"SynBioCrow_2.2.0-rc1_release_manifest.json")
print(json.dumps(report,indent=2,sort_keys=True))
print("acceptance report sha256",sha256(report_path))
print("acceptance bundle sha256",sha256(bundle))

drive.mount("/content/drive",force_remount=False)
out=Path("/content/drive/MyDrive/SynBioCrow/release/2.2.0-rc1/acceptance")
out.mkdir(parents=True,exist_ok=True)
for p in [report_path,bundle]:
 shutil.copy2(p,out/p.name)
 print("saved",out/p.name)
print("RC ACCEPTANCE: PASS")
print("Next release action: promote the accepted frozen commit to final v2.2.0 only after reviewing this report.")
